> 在线讲义：[第 24 章 · 收尾：全书地图、PPO 到 GRPO、接下来做什么](https://wikieden.github.io/rl-from-scratch/lessons/24-wrap-up.html) · 本书《从零学强化学习》

# 第 24 章 · 收尾：全书地图、PPO 到 GRPO、接下来做什么

本章回答：24 章学过的东西怎么串成一张图？LLM 训练里的 GRPO 和 PPO 差在哪？学完以后去做哪些作业、读哪些论文？

配套讲义：仓库里 `cs285-deep-rl/notes/lessons/24-wrap-up.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)  # 打印保留 3 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 代码 | 跑出来 |
|---|---|---|
| [1, 0, 0, 1] | `group_advantage([1, 0, 0, 1])` | [1, −1, −1, 1] |
| [1, 0, 0, 0] | `group_advantage([1, 0, 0, 0])` | [1.732, −0.577, −0.577, −0.577] |
| [1, 1, 1, 1] | `group_advantage([1, 1, 1, 1])` | [0, 0, 0, 0] |

## 1. 先把要用的东西定义好

这几格只是定义（环境、函数），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `group_advantage`**（来自 `ch24_wrap_up.py`）：下面要用到的小工具。

In [ ]:
def group_advantage(rewards, eps=1e-8):
    r = np.asarray(rewards, float)
    return (r - r.mean()) / (r.std() + eps)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
for rs in [[1, 0, 0, 1], [1, 0, 0, 0], [1, 1, 1, 1]]:
        print("一题 4 个回答的奖励", rs, "→ 优势", np.round(group_advantage(rs), 3))

## 3. 代码里才看得到的

全对时标准差是 0，直接除会得到 0 / 0 = nan（not a number，不是一个数），一个 nan 就能把整个模型的参数全变成 nan。所以分母一定要加一个很小的 eps（这里 $10^{-8}$）：分子也是 0，结果正好是 0。

## 4. 自己改一改

讲义里建议的几个小实验（数字是实跑出来的，改完对一对）：

1. 8 个回答里 2 个对：`group_advantage([1, 1, 0, 0, 0, 0, 0, 0])` → 对的 1.732，错的 −0.577。对的越少，答对的那几个优势越大。
2. 奖励不只是 0 / 1：`[5, 3, 1, 1]` → [1.508, 0.302, −0.905, −0.905]；把 [1, 0, 0, 1] 放大成 `[10, 0, 0, 10]`，结果还是 [1, −1, −1, 1]：除以标准差以后，奖励的单位不影响优势。

讲义末尾的练习题也可以在下面的空格子里用代码验算：先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 前半写成 MDP 和 Bellman，中间不知道 p 边试边学，后半价值、策略两条路；
> 递推、"旧 + α(目标 − 旧)"、偏差方差，三句话贯穿全书；
> LLM：上文是状态，下一个词是动作，一个回答是一局；
> GRPO：同题答几遍，组内平均当 baseline，全对全错没信号；KL 拴住参考模型，别学坏。